# Imputation Model Training

Trains and serialises an imputation pipeline to fill missing environmental variables in incoming farm profiles.

## Approach: IterativeImputer + Random Forest

Uses scikit-learn `IterativeImputer` with `RandomForestRegressor` as the base estimator.
Each feature is modelled as a function of all others and the process iterates to convergence.


## Features

| Column | Type | Notes |
|---|---|---|
| `elevation_m` | Numeric | |
| `slope` | Numeric | |
| `temperature_celsius` | Numeric | |
| `rainfall_mm` | Numeric | |
| `ph` | Numeric | |


Base features available from farm geometry: `latitude`, `longitude`, `area_ha`, `coastal`, `riparian`.

## Dataset

`backend/src/scripts/data/farm_master.csv`

## 1. Imports

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DATA_PATH = Path("../../backend/src/scripts/data/farm_master.csv")
MODELS_PATH = Path("../src/models/imputation")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print()
print(df.dtypes)
df.head()

Shape: (3200, 15)
Nulls: 0

external_id              int64
ph                     float64
rainfall_mm              int64
temperature_celsius      int64
elevation_m              int64
area_ha                float64
longitude              float64
latitude               float64
soil_texture_id          int64
coastal                   bool
slope                  float64
riparian                  bool
nitrogen_fixing           bool
shade_tolerant            bool
bank_stabilising          bool
dtype: object


,external_id,ph,rainfall_mm,temperature_celsius,elevation_m,area_ha,longitude,latitude,soil_texture_id,coastal,slope,riparian,nitrogen_fixing,shade_tolerant,bank_stabilising
0,1,6.2,1959,24,585,0.36,126.67570,-8.56880,12,False,13.11,False,False,False,False
1,2,6.2,1959,24,481,0.48,126.68015,-8.56758,12,False,13.94,False,False,False,False
2,3,8.2,2021,25,179,1.18,126.66465,-8.64226,3,False,11.15,False,False,False,False
3,4,5.9,2554,24,260,0.46,126.65146,-8.63969,3,False,15.55,False,False,False,False
4,5,7.0,2021,25,129,2.00,126.67008,-8.64300,12,False,7.62,True,False,False,False


## 3. Prepare Features

In [3]:
# Base features — always available from farm geometry
BASE_FEATURES = ["latitude", "longitude", "area_ha", "coastal", "riparian"]

# Target features — numeric environmental variables that may be missing
TARGET_FEATURES = ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "ph"]
ALL_FEATURES = BASE_FEATURES + TARGET_FEATURES

# Prepare feature matrix — convert booleans to int for sklearn
X_full = df[ALL_FEATURES].copy()
X_full["coastal"] = X_full["coastal"].astype(int)
X_full["riparian"] = X_full["riparian"].astype(int)

print(f"Feature matrix shape: {X_full.shape}")
X_full.head()

Feature matrix shape: (3200, 10)


,latitude,longitude,area_ha,coastal,riparian,elevation_m,slope,temperature_celsius,rainfall_mm,ph
0,-8.56880,126.67570,0.36,0,0,585,13.11,24,1959,6.2
1,-8.56758,126.68015,0.48,0,0,481,13.94,24,1959,6.2
2,-8.64226,126.66465,1.18,0,0,179,11.15,25,2021,8.2
3,-8.63969,126.65146,0.46,0,0,260,15.55,24,2554,5.9
4,-8.64300,126.67008,2.00,0,1,129,7.62,25,2021,7.0


## 4. Fit Imputation Pipeline on Full Dataset

In [4]:
# RandomForestRegressor does not require feature scaling
final_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
    max_iter=10,
    random_state=RANDOM_SEED,
)
final_imputer.fit(X_full)
print("Imputer fitted on full dataset.")

Imputer fitted on full dataset.


## 5. Evaluate Pipeline

Since the dataset has no real missing values, evaluation uses **MCAR masking** (missing completely at random)
with **5-fold cross-validation**. In each fold:
1. Fit the imputer on the training split
2. Mask 30% of target values in the test split
3. Impute and measure recovery in original units

Metrics are averaged across all folds.

In [5]:
rng = np.random.default_rng(RANDOM_SEED)
MASK_RATE = 0.30
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

fold_results = {col: [] for col in TARGET_FEATURES}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_full), 1):
    X_train = X_full.iloc[train_idx]
    X_test = X_full.iloc[test_idx].copy()

    imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
        max_iter=10,
        random_state=RANDOM_SEED,
    )
    imputer.fit(X_train)

    true_values = {}
    for col in TARGET_FEATURES:
        mask = rng.random(len(X_test)) < MASK_RATE
        true_values[col] = (X_test.loc[X_test.index[mask], col].values.copy(), mask)
        X_test.loc[X_test.index[mask], col] = np.nan

    # RF imputer outputs original-scale values directly — no inverse_transform needed
    X_imp = imputer.transform(X_test)
    X_imp_df = pd.DataFrame(X_imp, columns=X_full.columns, index=X_test.index)

    for col in TARGET_FEATURES:
        y_true, mask = true_values[col]
        y_pred = X_imp_df.loc[X_test.index[mask], col].values
        if len(y_true) == 0:
            continue
        fold_results[col].append(np.sqrt(mean_squared_error(y_true, y_pred)))

    print(f"Fold {fold}/{N_SPLITS} done")

Fold 1/5 done


Fold 2/5 done


Fold 3/5 done


Fold 4/5 done


Fold 5/5 done


In [6]:
# Suggested RMSE thresholds based on domain context and prior model performance
THRESHOLDS = {
    "elevation_m": 100.0,  # metres  — GEE SRTM MAE ~11m; 100m reflects fallback tolerance
    "slope": 5.0,  # degrees — acceptable planting slope error
    "temperature_celsius": 2.0,  # °C      — agronomic species sensitivity
    "rainfall_mm": 200.0,  # mm      — species rainfall band width
    "ph": 1.0,  # pH unit — typical soil pH tolerance range
}

print("=" * 72)
print(f"EVALUATION SUMMARY — {N_SPLITS}-fold CV, {MASK_RATE:.0%} MCAR masking")
print("=" * 72)
header = f"{'Feature':25s} | {'RMSE (mean±std)':>18} | {'Threshold':>10} | {'Status'}"
print(header)
print("-" * 72)
for col in TARGET_FEATURES:
    scores = fold_results[col]
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    threshold = THRESHOLDS[col]
    status = "PASS" if mean_score <= threshold else "FAIL"
    rmse_str = f"{mean_score:.3f} ± {std_score:.3f}"
    print(f"{col:25s} | {rmse_str:>18} | {threshold:>10.1f} | {status}")

EVALUATION SUMMARY — 5-fold CV, 30% MCAR masking
Feature                   |    RMSE (mean±std) |  Threshold | Status
------------------------------------------------------------------------
elevation_m               |    123.556 ± 8.562 |      100.0 | FAIL
slope                     |      4.642 ± 0.245 |        5.0 | PASS
temperature_celsius       |      0.685 ± 0.088 |        2.0 | PASS
rainfall_mm               |    47.938 ± 15.384 |      200.0 | PASS
ph                        |      0.434 ± 0.039 |        1.0 | PASS


## 6. Save Model Artefacts

> **Trained model saved to**
 `src/models/imputation/`.

In [7]:
joblib.dump(final_imputer, MODELS_PATH / "imputation_pipeline.joblib", compress=("lzma", 9))
joblib.dump(list(X_full.columns), MODELS_PATH / "feature_columns.joblib")

print("Saved:")
for f in sorted(MODELS_PATH.glob("*.joblib")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:6.1f} KB")

Saved:
  feature_columns.joblib                           0.1 KB
  imputation_pipeline.joblib                    18295.3 KB
